# 센서 구간 분류·회귀·이상탐지

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

입력은 window_id·subject_id·timestamp·센서 채널의 long 표입니다. 한 구간에 하나의 정답을 둡니다. 전체 구간 관측 후 판정하는 문제용이며 실시간 조기 판정에 그대로 사용하지 마세요.

In [ ]:
DEMO=True
TRAIN_PATH='data/sensor_train.csv';TEST_PATH='data/sensor_test.csv'
WINDOW='window_id';GROUP='subject_id';TIME_COL='timestamp';TARGET='target'
CHANNELS=['ax','ay','az'];TASK='classification' # classification / regression / anomaly
METRIC='f1_macro';SAMPLE_PATH=None
SAMPLING_HZ=20.0;USE_FFT=True
OUTPUT='outputs/time_series/sensor_submission.csv'


## 공통 함수

그룹 분리와 결측 처리

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 데이터

겹치는 구간과 같은 장비/사람은 반드시 같은 GROUP으로 묶습니다.

In [ ]:
def demo_windows(subjects,has_target=True):
    records=[]
    for subject in subjects:
        for w in range(4):
            label=(subject+w)%2
            for j in range(32):
                row={WINDOW:f'{subject}_{w}',GROUP:subject,TIME_COL:pd.Timestamp('2025-01-01')+pd.Timedelta(seconds=(w*32+j)/SAMPLING_HZ)}
                for k,c in enumerate(CHANNELS): row[c]=np.sin(2*np.pi*(label+1)*j/SAMPLING_HZ)+label*.8+rng.normal(0,.1)
                if has_target: row[TARGET]=float(label)+subject*.01 if TASK=='regression' else label
                records.append(row)
    return pd.DataFrame(records)
if DEMO: train=demo_windows(range(12));test=demo_windows(range(20,23),False)
else: train=read_table(TRAIN_PATH);test=read_table(TEST_PATH)
for frame in (train,test):
    frame[TIME_COL]=pd.to_datetime(frame[TIME_COL],errors='raise')
    if frame[[WINDOW,GROUP,TIME_COL]].isna().any().any(): raise ValueError('구간·개체·시간 결측')
    if frame.duplicated([WINDOW,TIME_COL]).any(): raise ValueError('구간 내 중복 시각')
if TASK!='anomaly' and train[TARGET].isna().any(): raise ValueError('정답 결측')


## 통계·주파수 특징과 학습·예측

구간별 통계는 독립 계산하고, 구간 간 대치/정규화는 학습 그룹만 사용합니다. 그룹 홀드아웃은 새로운 개체 일반화 평가입니다. 회귀 시 METRIC도 바꾸세요.

In [ ]:
def window_features(frame,labels=False):
    records=[]
    for wid,g in frame.groupby(WINDOW,sort=False):
        g=g.sort_values(TIME_COL)
        if g[GROUP].nunique()!=1: raise ValueError('한 window에 여러 subject가 있습니다.')
        row={WINDOW:wid,GROUP:g[GROUP].iloc[0]}
        if labels:
            if g[TARGET].nunique()!=1: raise ValueError('구간별 정답은 하나여야 합니다.')
            row[TARGET]=g[TARGET].iloc[0]
        if USE_FFT:
            dt=g[TIME_COL].diff().dropna().dt.total_seconds().to_numpy()
            if len(dt)<2 or not np.allclose(dt,1/SAMPLING_HZ,rtol=.02,atol=1e-6): raise ValueError('FFT는 설정한 Hz의 등간격 데이터만 지원합니다.')
        for c in CHANNELS:
            values=pd.to_numeric(g[c],errors='raise').replace([np.inf,-np.inf],np.nan).to_numpy(dtype=float)
            good=values[np.isfinite(values)];row[f'{c}_missing']=1-len(good)/len(values)
            if not len(good):
                for stat in ('mean','std','min','max','rms','delta','dominant_hz'):row[f'{c}_{stat}']=np.nan
                continue
            row.update({f'{c}_mean':good.mean(),f'{c}_std':good.std(),f'{c}_min':good.min(),f'{c}_max':good.max(),f'{c}_rms':np.sqrt(np.mean(good**2)),f'{c}_delta':good[-1]-good[0]})
            row[f'{c}_dominant_hz']=np.nan
            if USE_FFT and len(good)==len(values):
                spectrum=abs(np.fft.rfft(values-values.mean()));freq=np.fft.rfftfreq(len(values),d=1/SAMPLING_HZ)
                row[f'{c}_dominant_hz']=freq[1+np.argmax(spectrum[1:])]
        records.append(row)
    return pd.DataFrame(records)
features=window_features(train,TASK!='anomaly');future=window_features(test)
columns=[c for c in features if c not in (WINDOW,GROUP,TARGET)]
X=features[columns];Xt=future[columns]
if TASK=='anomaly':
    final_model=make_pipeline(tabular_preprocessor(X),IsolationForest(contamination=.05,n_estimators=80,random_state=SEED,n_jobs=1)).fit(X)
    predictions=-final_model[-1].score_samples(final_model[:-1].transform(Xt));outcol='anomaly_score'
else:
    a,b=split_rows(features,TARGET,TASK,'group',GROUP)
    y=features[TARGET];prep=tabular_preprocessor(X.iloc[a])
    if TASK=='classification': base={'linear':LogisticRegression(max_iter=500,class_weight='balanced'),'trees':ExtraTreesClassifier(n_estimators=60,random_state=SEED,n_jobs=1)}
    elif TASK=='regression': base={'linear':Ridge(alpha=10,solver='lsqr'),'trees':ExtraTreesRegressor(n_estimators=60,random_state=SEED,n_jobs=1)}
    else: raise ValueError(TASK)
    candidates={k:make_pipeline(clone(prep),v) for k,v in base.items()}
    best,validation_model,results=fit_compare(candidates,X,y,a,b,TASK,METRIC)
    final_model=clone(candidates[best]).fit(X,y);predictions=final_model.predict(Xt);outcol=TARGET
submission=write_submission(future[WINDOW],predictions,WINDOW,[outcol],OUTPUT,SAMPLE_PATH)
